# Авито: кандидатогенерация — предобработка

Что делает ноутбук (по порядку):

1. Загружает parquet и сразу чинит типы: `decimal128` → `float64`
2. Удаляет бесполезную колонку `search_is_delivery_search`
3. Раскладывает train на три таблицы: **запросы**, **объявления**, **пары** (запрос → выбранное объявление)
4. Выделяет валидацию из train: запросы, чьи выбранные объявления есть в корпусе бенчмарка
5. Чистит числовые признаки (цена, рейтинг, отзывы, флаги)
6. Нормализует тексты: одна версия для нейросетей, одна для лемматизации
7. Лемматизирует через pymorphy3 с кэшем по словарю (каждое уникальное слово — один раз)
8. Разбирает фильтры поиска на сегменты и размечает объявления теми же сегментами
9. Считает гео: оценку координат локации поиска, в том числе для «региональных» id без своих объявлений
10. Печатает диагностику: насколько безопасны фильтры по категории, фильтрам и гео
11. Сохраняет всё в `/kaggle/working/prep/`

**Kaggle:** для установки pymorphy3 нужен интернет (Settings → Internet → On). GPU здесь не нужен.

### Установка pymorphy3 (лемматизатор для русского языка, open-source, работает локально)

In [1]:
!pip install -q pymorphy3 pymorphy3-dicts-ru

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 61.5 MB/s eta 0:00:00


### Импорты и настройки

In [2]:
import os, re, gc, glob, time, warnings
from collections import Counter
from multiprocessing import Pool

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 250)
pd.set_option('display.max_colwidth', 120)

SEED = 42
N_JOBS = os.cpu_count() or 2                       # на Kaggle 4 ядра
OUT_DIR = '/kaggle/working/prep' if os.path.isdir('/kaggle/working') else './prep'
os.makedirs(OUT_DIR, exist_ok=True)
print('ядер:', N_JOBS, '| результаты ->', OUT_DIR)

DIAG = {}   # ключевые цифры диагностики, выведем в конце

ядер: 4 | результаты -> /kaggle/working/prep


## 1. Загрузка с исправлением типов

EDA показал, что цена и координаты лежат в parquet как `decimal128`. Pandas превращает их в объекты Python `Decimal`:
медленно, много памяти, и половина numpy-операций на них не работает. Приводим такие колонки к `float64` ещё на уровне arrow,
до перехода в pandas. `large_string` в бенчмарке приводим к обычному `string`, чтобы типы во всех файлах совпадали.

In [3]:
def find_file(name, roots=('/kaggle/input', '.')):
    """Рекурсивно ищет файл в /kaggle/input (там лежит подключённый датасет) или в текущей папке."""
    for root in roots:
        hits = glob.glob(os.path.join(root, '**', name), recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(name)

def read_parquet_clean(path):
    """Читает parquet и приводит типы до перехода в pandas: decimal -> float64, large_string -> string."""
    table = pq.read_table(path)
    fields = []
    for f in table.schema:
        if pa.types.is_decimal(f.type):
            fields.append(pa.field(f.name, pa.float64()))
        elif pa.types.is_large_string(f.type):
            fields.append(pa.field(f.name, pa.string()))
        else:
            fields.append(f)
    df = table.cast(pa.schema(fields)).to_pandas()
    print(f'{os.path.basename(path):<28} {df.shape}')
    return df

t0 = time.time()
train  = read_parquet_clean(find_file('train.parquet'))
bench  = read_parquet_clean(find_file('benchmark_queries.parquet'))
corpus = read_parquet_clean(find_file('benchmark_items.parquet'))
print(f'загружено за {time.time() - t0:.0f} с')
print(train[['item_price', 'item_latitude', 'item_longitude']].dtypes)

train.parquet                (497673, 19)
benchmark_queries.parquet    (2452, 6)
benchmark_items.parquet      (189212, 14)
загружено за 24 с
item_price        float64
item_latitude     float64
item_longitude    float64
dtype: object


## 2. Удаление `search_is_delivery_search`

В бенчмарке эта колонка всегда равна 0, в train значение 1 встречается в 12 строках из 497 тыс.
Признак, который на тесте константа, ничего не даёт, поэтому удаляем.

In [4]:
DROP_COLS = ['search_is_delivery_search']
print('значения в бенчмарке:', bench['search_is_delivery_search'].value_counts().to_dict())
train  = train.drop(columns=DROP_COLS)
bench  = bench.drop(columns=DROP_COLS)
print('удалено:', DROP_COLS)

значения в бенчмарке: {0: 2452}
удалено: ['search_is_delivery_search']


## 3. Train → три таблицы: запросы, объявления, пары

Сейчас в train одно объявление повторяется во многих строках (345 тыс. уникальных на 498 тыс. строк),
а его описание в среднем занимает 1400 символов. Если лемматизировать прямо в train, одни и те же тексты обработаются много раз. Поэтому раскладываем данные так:

- **queries**: уникальные запросы train (псевдо-запрос = текст + локация + категория + фильтры) плюс запросы бенчмарка
- **items**: уникальные объявления из корпуса и из train, флаг `in_corpus` показывает, есть ли объявление в корпусе бенчмарка
- **pairs**: `qid → item_id` плюс сколько раз встретилась пара

In [5]:
KEY_COLS = ['search_query', 'search_location_id', 'search_category', 'search_infm_params_text']

for df in (train, bench):
    # пустые фильтры бывают и NaN, и '' -> приводим к '', иначе ключи train и бенчмарка не совпадут
    df['search_infm_params_text'] = df['search_infm_params_text'].fillna('').astype(str).str.strip()
    df['search_query'] = df['search_query'].fillna('').astype(str)

# id псевдо-запроса train: хэш всех полей запроса
h = pd.util.hash_pandas_object(train[KEY_COLS].astype(str), index=False).values
train['qid'] = ['tr_' + format(x, '016x') for x in h]

queries = pd.concat([
    train.drop_duplicates('qid')[['qid'] + KEY_COLS].assign(split='train'),   # split уточним в п.4
    bench.rename(columns={'query_id': 'qid'})[['qid'] + KEY_COLS].assign(split='bench'),
], ignore_index=True)

pairs = train.groupby(['qid', 'item_id'], as_index=False).size().rename(columns={'size': 'n_rows'})

ITEM_COLS = list(corpus.columns)
corpus_ids = set(corpus['item_id'])
# из train берём объявления, которых нет в корпусе; keep='last' = самая свежая версия признаков
train_only = (train[ITEM_COLS].drop_duplicates('item_id', keep='last')
              .loc[lambda d: ~d['item_id'].isin(corpus_ids)])
items = pd.concat([corpus.assign(in_corpus=True), train_only.assign(in_corpus=False)], ignore_index=True)
pairs['in_corpus'] = pairs['item_id'].isin(corpus_ids)

print(f'queries: {len(queries):,} (train {(queries.split == "train").sum():,}, bench {(queries.split == "bench").sum():,})')
print(f'items:   {len(items):,} (в корпусе {items.in_corpus.sum():,})')
print(f'pairs:   {len(pairs):,} (с объявлением из корпуса {pairs.in_corpus.sum():,})')

del train, bench, corpus, train_only; _ = gc.collect()

queries: 356,914 (train 354,462, bench 2,452)
items:   515,895 (в корпусе 189,212)
pairs:   467,054 (с объявлением из корпуса 29,572)


## 4. Валидация из train

С 7 попытками без локальной валидации никак. Хорошая новость: 6.6% пар train ведут на объявления, которые есть в корпусе бенчмарка.
По таким запросам можно честно посчитать Recall@50 **на настоящем корпусе из 189 тыс. объявлений**, то есть в тех же условиях, что и бенчмарк.

Правила разбиения:
- в val попадают запросы, у которых хотя бы одно выбранное объявление есть в корпусе
- релевантные для val = выбранные объявления, которые есть в корпусе (остальные найти невозможно)
- делим **по тексту запроса**: если текст ушёл в val, все запросы с этим текстом убираются из обучения. В бенчмарке все 2452 текста уникальны, и только 37% из них встречаются в train, поэтому утечка через одинаковые тексты дала бы слишком оптимистичную оценку
- на один текст берём один запрос, как в бенчмарке
- берём только редкие тексты (не больше 5 запросов в train): бенчмарк состоит из длинных уникальных запросов, и так мы меньше теряем обучающих данных

In [6]:
VAL_SIZE = 3000
rng = np.random.default_rng(SEED)

queries['query_key'] = (queries['search_query'].str.lower().str.replace('ё', 'е')
                        .str.replace(r'\s+', ' ', regex=True).str.strip())

tr_q = queries[queries.split == 'train']
q_with_corpus = set(pairs.loc[pairs.in_corpus, 'qid'])
cand = tr_q[tr_q.qid.isin(q_with_corpus)]

# случайные тексты из кандидатов, по одному запросу на текст
# Берём только редкие тексты (<= MAX_TEXT_FREQ запросов в train). Во-первых, бенчмарк состоит из длинных
# уникальных запросов. Во-вторых, если в val уйдёт «маникюр», из обучения пришлось бы выкинуть тысячи запросов.
MAX_TEXT_FREQ = 5
text_freq = tr_q['query_key'].value_counts()
cand_texts = cand['query_key'].unique()
cand_texts = cand_texts[text_freq.reindex(cand_texts).values <= MAX_TEXT_FREQ]
val_texts = set(rng.choice(cand_texts, size=min(VAL_SIZE, len(cand_texts)), replace=False))
val_qids = (cand[cand.query_key.isin(val_texts)]
            .sample(frac=1, random_state=SEED).drop_duplicates('query_key')['qid'])

is_train_q = queries.split == 'train'
queries.loc[is_train_q & queries.qid.isin(set(val_qids)), 'split'] = 'val'
# остальные запросы с теми же текстами выкидываем из обучения, чтобы не было утечки
queries.loc[is_train_q & queries.query_key.isin(val_texts) & (queries.split == 'train'), 'split'] = 'drop'

pairs = pairs.merge(queries[['qid', 'split']], on='qid', how='left')
print(queries.split.value_counts())
print(f'длина запроса в словах: val {queries.loc[queries.split == "val", "query_key"].str.split().str.len().mean():.2f}, '
      f'bench {queries.loc[queries.split == "bench", "query_key"].str.split().str.len().mean():.2f}, '
      f'train {queries.loc[queries.split == "train", "query_key"].str.split().str.len().mean():.2f}')
val_pos = pairs[(pairs.split == 'val') & pairs.in_corpus].groupby('qid').size()
print(f'\nval: {len(val_pos):,} запросов, релевантных на запрос: среднее {val_pos.mean():.2f}, макс {val_pos.max()}')
DIAG['val: запросов'] = len(val_pos)

split
train    348295
drop       3167
val        3000
bench      2452
Name: count, dtype: int64
длина запроса в словах: val 3.12, bench 3.20, train 2.45

val: 3,000 запросов, релевантных на запрос: среднее 1.02, макс 4


## 5. Числовые признаки объявлений

- **цена**: в EDA среднее 2e7 при медиане 1000, то есть есть выбросы и заглушки (1 руб. = «цена не указана»). Делаем `log1p` с обрезкой по 99.9 перцентилю и флаг «символическая цена»
- **рейтинг**: пропуски оставляем как NaN (GBDT с ними работает) и добавляем флаг «нет рейтинга»
- **отзывы**: пропуск = 0 отзывов, берём `log1p`
- **bool-флаги** → int8

In [7]:
price = items['item_price']
print('цена, квантили:', price.quantile([.01, .1, .5, .9, .99, .999]).round(0).to_dict())
print(f'доля цены <= 10 руб.: {(price <= 10).mean():.2%}')

items['price_log'] = np.log1p(price.clip(lower=0, upper=price.quantile(0.999))).astype('float32')
items['price_is_symbolic'] = (price <= 10).astype('int8')
items['rating_isna'] = items['item_rating'].isna().astype('int8')
items['item_rating'] = items['item_rating'].astype('float32')
items['reviews_log'] = np.log1p(items['item_rating_reviews_count'].fillna(0)).astype('float32')
for c in ['item_is_phone_hidden', 'item_is_message_forbidden']:
    items[c] = items[c].astype('int8')
items['lat'] = items['item_latitude'].astype('float32')
items['lon'] = items['item_longitude'].astype('float32')
items = items.drop(columns=['item_price', 'item_rating_reviews_count', 'item_latitude', 'item_longitude'])
print('готово:', [c for c in items.columns if c not in ('item_title_raw', 'item_description_raw', 'item_infm_params_text')])

цена, квантили: {0.01: -1.0, 0.1: 10.0, 0.5: 1000.0, 0.9: 5500.0, 0.99: 100000.0, 0.999: 4981641.0}
доля цены <= 10 руб.: 10.10%
готово: ['item_rating', 'item_microcat_id', 'item_location_id', 'item_is_phone_hidden', 'item_is_message_forbidden', 'item_id', 'item_category_id', 'in_corpus', 'price_log', 'price_is_symbolic', 'rating_isna', 'reviews_log', 'lat', 'lon']


## 6. Нормализация текстов

Делаем **две версии** каждого текста, потому что они нужны для разного:

- `*_norm`: мягкая чистка для нейросетей (эмбеддингов). Удаляем ссылки, телефоны, email, эмодзи и буллеты, литеральные `\n`, меняем ё→е.
  Регистр и пунктуацию оставляем: трансформеры обучены на обычном тексте, лемматизация им только мешает
- `*_tok`: нижний регистр, только буквы и цифры. Это вход для лемматизации (п.7), потом из неё получится `*_lem` для BM25

Обрабатываем параллельно на всех ядрах.

In [8]:
URL_RE   = re.compile(r'https?://\S+|www\.\S+')
EMAIL_RE = re.compile(r'\S+@\S+\.\S+')
PHONE_RE = re.compile(r'(?:\+7|\b8)[\s\-(]*\d{3}[\s\-)]*\d{3}[\s\-]*\d{2}[\s\-]*\d{2}')
NOISE_RE = re.compile(r'[^\w\s.,:;!?()«»"\'%№/+\-]')      # эмодзи, ●, ✅ и прочие символы
WS_RE    = re.compile(r'\s+')
NONALNUM_RE = re.compile(r'[^a-zа-я0-9]+')

def normalize_for_model(s):
    """Мягкая чистка: текст остаётся читаемым, для трансформеров."""
    if not isinstance(s, str) or not s:
        return ''
    s = s.replace('\\n', ' ').replace('ё', 'е').replace('Ё', 'Е')
    s = URL_RE.sub(' ', s)
    s = EMAIL_RE.sub(' ', s)
    s = PHONE_RE.sub(' ', s)
    s = NOISE_RE.sub(' ', s)
    return WS_RE.sub(' ', s).strip()

def to_tokens(s):
    """Жёсткая чистка под лемматизацию: только [a-zа-я0-9], слова через пробел."""
    return WS_RE.sub(' ', NONALNUM_RE.sub(' ', s.lower())).strip()

def _prep_chunk(texts):
    out = []
    for t in texts:
        n = normalize_for_model(t)
        out.append((n, to_tokens(n)))
    return out

def parallel_map(func, seq, chunk=20_000):
    """Делит список на куски и обрабатывает их в N_JOBS процессах."""
    seq = list(seq)
    chunks = [seq[i:i + chunk] for i in range(0, len(seq), chunk)]
    with Pool(N_JOBS) as p:
        res = p.map(func, chunks)
    return [x for r in res for x in r]

def prep_column(df, src, prefix):
    t = time.time()
    res = parallel_map(_prep_chunk, df[src].tolist())
    df[f'{prefix}_norm'] = [r[0] for r in res]
    df[f'{prefix}_tok']  = [r[1] for r in res]
    print(f'{src:<24} -> {prefix}_norm / {prefix}_tok  ({time.time() - t:.0f} с)')

prep_column(queries, 'search_query', 'query')
prep_column(items, 'item_title_raw', 'title')
prep_column(items, 'item_infm_params_text', 'params')
prep_column(items, 'item_description_raw', 'desc')

MAX_DESC_CHARS = 4000     # для нейросетей длиннее всё равно не влезет (512 токенов); лемматизация идёт по полному тексту
items['desc_norm'] = items['desc_norm'].str.slice(0, MAX_DESC_CHARS)

search_query             -> query_norm / query_tok  (3 с)
item_title_raw           -> title_norm / title_tok  (6 с)
item_infm_params_text    -> params_norm / params_tok  (124 с)
item_description_raw     -> desc_norm / desc_tok  (151 с)


## 7. Лемматизация с кэшем

pymorphy3 разбирает одно слово за раз и работает медленно. Но уникальных слов намного меньше, чем словоупотреблений:
в описаниях сотни миллионов токенов, а словарь на порядки меньше. Поэтому:

1. собираем словарь всех уникальных токенов
2. лемматизируем каждое слово **один раз** (параллельно)
3. заменяем слова на леммы через словарь и выкидываем стоп-слова

Числа и латиницу (`lexus`, `rx350`, `24`) не трогаем: pymorphy для них бесполезен, а для поиска они важны.

In [9]:
TOK_COLS = [(queries, 'query'), (items, 'title'), (items, 'params'), (items, 'desc')]

def _count_chunk(texts):
    c = Counter()
    for t in texts:
        c.update(t.split())
    return [c]

t = time.time()
vocab = Counter()
for df, p in TOK_COLS:
    for c in parallel_map(_count_chunk, df[f'{p}_tok'].tolist(), chunk=50_000):
        vocab.update(c)
print(f'уникальных токенов: {len(vocab):,}, всего словоупотреблений: {sum(vocab.values()):,} ({time.time() - t:.0f} с)')

уникальных токенов: 1,008,527, всего словоупотреблений: 170,872,109 (46 с)


In [10]:
CYR_RE = re.compile(r'[а-я]')

def _lemmatize_words(words):
    import pymorphy3
    morph = pymorphy3.MorphAnalyzer()
    return [(w, morph.parse(w)[0].normal_form.replace('ё', 'е')) for w in words]

t = time.time()
cyr_words = [w for w in vocab if CYR_RE.search(w) and len(w) > 2]
LEMMA = {w: w for w in vocab}                          # по умолчанию слово = своя лемма
for w, l in parallel_map(_lemmatize_words, cyr_words, chunk=50_000):
    LEMMA[w] = l
print(f'лемматизировано {len(cyr_words):,} слов за {time.time() - t:.0f} с')
print('примеры:', {w: LEMMA[w] for w in ['телевизоров', 'видеодомофонов', 'дровах', 'наращивание', 'ресниц'] if w in LEMMA})

лемматизировано 826,725 слов за 80 с
примеры: {'телевизоров': 'телевизор', 'видеодомофонов': 'видеодомофон', 'дровах': 'дрова', 'наращивание': 'наращивание', 'ресниц': 'ресница'}


In [11]:
# Короткий стоп-лист (без внешних загрузок вроде nltk, чтобы решение воспроизводилось офлайн).
# Числительные и «без» не выкидываем: «без предоплаты», «2 комнаты» — это смысл.
STOPWORDS = set("""
и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее мне было вот от меня
еще нет о из ему теперь когда даже ну ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом
себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтобы будто чего раз тоже себе под будет
ж тогда кто этот того потому этого какой совсем ним здесь этом почти мой тем нее сейчас были куда зачем всех никогда
можно при наконец об хоть после над через эти нас про всего них какая много разве эту моя впрочем свою этой перед
иногда чуть том нельзя такой им более всегда конечно всю между весь свой который это наш ваш
""".split())

def _apply_lemmas(texts):
    return [' '.join(l for l in (LEMMA[w] for w in t.split()) if l not in STOPWORDS) for t in texts]

for df, p in TOK_COLS:
    t = time.time()
    df[f'{p}_lem'] = parallel_map(_apply_lemmas, df[f'{p}_tok'].tolist())
    print(f'{p}_lem готово ({time.time() - t:.0f} с)')

display(queries[['search_query', 'query_norm', 'query_lem']].head(5))
display(items[['title_norm', 'title_lem']].head(5))

query_lem готово (2 с)
title_lem готово (3 с)
params_lem готово (25 с)
desc_lem готово (30 с)


,search_query,query_norm,query_lem
0,скупка телевизоров,скупка телевизоров,скупка телевизор
1,автоподбор,автоподбор,автоподбор
2,баня на дровах,баня на дровах,баня дрова
3,изготовление госномера на авто,изготовление госномера на авто,изготовление госномер авто
4,укладка плитки,укладка плитки,укладка плитка


,title_norm,title_lem
0,Ремонт/выкуп компьют. и ноутбуков с выездом на дом,ремонт выкуп компить ноутбук выезд дом
1,Обучение ребенка чтению,обучение ребенок чтение
2,Афрокудри 5+,афрокудри 5
3,Монтаж малых архитектурных форм,монтаж малый архитектурный форма
4,Репетитор по истории 10 класс ЕГЭ,репетитор история 10 класс егэ


## 8. Признак «удалённая услуга»

В EDA около 12% выбранных объявлений находятся дальше 300 км от места поиска. Гипотеза: это услуги,
которые оказываются удалённо (онлайн-обучение, обзвон, дизайн) или по всей стране (перевозки).
Помечаем такие объявления по тексту, а в п.10 проверим, объясняет ли флаг дальние выборы.

In [12]:
# «онлайн запись» — это способ записи, а не удалённая услуга; whatsapp почти в каждом объявлении, его не берём
REMOTE_RE = re.compile(r'онлайн(?! запис)|удаленн|дистанцион|по всей росси|по россии|по рф|zoom|skype|скайп|межгород')

items['is_remote_text'] = [
    int(bool(REMOTE_RE.search(a) or REMOTE_RE.search(b) or REMOTE_RE.search(c)))
    for a, b, c in zip(items['title_tok'], items['params_tok'], items['desc_tok'])
]
print(f'доля объявлений с признаками удалёнки: {items.is_remote_text.mean():.2%}')

# *_tok больше не нужны
items = items.drop(columns=['title_tok', 'params_tok', 'desc_tok'])
queries = queries.drop(columns=['query_tok'])
_ = gc.collect()

доля объявлений с признаками удалёнки: 29.46%


## 9. Фильтры поиска → сегменты, объявления → те же сегменты

Лемматизировать фильтры не нужно: это не свободный текст, а **структурированные значения** из таксономии Авито.
Фильтр `Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье` состоит из двух сегментов,
и в параметрах выбранного объявления они встречаются дословно. Сравнивать лучше точным совпадением, а не по леммам.

Порядок:
1. режем каждый фильтр на сегменты по началу `Вид услуги` / `Тип услуги`
2. из всех фильтров train и бенчмарка собираем словарь сегментов
3. находим в параметрах каждого объявления сегменты из словаря и сохраняем как теги
4. остальные части фильтра (`Онлайн-запись`, `Рейтинг пользователя 4 звезды и выше` и т.п.) складываем в `f_other` и проверяем отдельно

In [13]:
SEG_SPLIT = re.compile(r'(?=(?:Вид услуги|Тип услуги)\b)')
TAX_PREFIXES = ('Вид услуги', 'Тип услуги')

def split_filter(f):
    parts = [p.strip() for p in SEG_SPLIT.split(f) if p.strip()]
    # сегмент без значения («Вид услуги» целиком) ничего не ограничивает — выкидываем
    tax = [p for p in parts if p.startswith(TAX_PREFIXES) and p not in TAX_PREFIXES]
    other = [p for p in parts if not p.startswith(TAX_PREFIXES)]
    return tax, other

sp = [split_filter(f) for f in queries['search_infm_params_text']]
queries['f_tax'] = ['|'.join(t) for t, _ in sp]
queries['f_other'] = [' '.join(o) for _, o in sp]

TAX_SEGMENTS = sorted({s for t, _ in sp for s in t}, key=len, reverse=True)
print(f'уникальных сегментов таксономии: {len(TAX_SEGMENTS)}')
print('примеры:', TAX_SEGMENTS[:5])
print('\nне-таксономические части фильтров в бенчмарке:')
display(queries.loc[queries.split == 'bench', 'f_other'].replace('', np.nan).value_counts().head(15))

уникальных сегментов таксономии: 934
примеры: ['Тип услуги Дизайн интерьеров Какими жилыми помещениями занимаетесь Квартира Какими комнатами занимаетесь Ванная Какими комнатами занимаетесь Санузел Какими комнатами занимаетесь Балкон, лоджия Какими комнатами занимаетесь Коридор Какими комнатами занимаетесь Прихожая Какими комнатами занимаетесь Гардеробная Какими комнатами занимаетесь Кабинет Какими комнатами занимаетесь Гостиная Какими комнатами занимаетесь Детская Какими комнатами занимаетесь Спальня Какими комнатами занимаетесь Кухня В каких стилях работаете лофт В каких стилях работаете современный В каких стилях работаете классика Кто оказывает услуги Частный исполнитель', 'Тип услуги Дизайн интерьеров Какими жилыми помещениями занимаетесь Дом Какими комнатами занимаетесь Кухня Какими комнатами занимаетесь Спальня Какими комнатами занимаетесь Детская Какими комнатами занимаетесь Гостиная Какими комнатами занимаетесь Кабинет Какими комнатами занимаетесь Гардеробная Какими комнатами з

f_other
Онлайн-запись                                                                                                                                                                                     17
Кто оказывает услуги Частный исполнитель                                                                                                                                                          11
Предмет или специальность Английский язык                                                                                                                                                          7
Кто оказывает услуги Частный исполнитель Кто оказывает услуги Компания                                                                                                                             4
Предмет или специальность Испанский язык                                                                                                                                                           2
Срочная

In [14]:
# Регулярка «префикс + одно из известных значений», значение должно заканчиваться на границе следующего
# параметра (пробел + заглавная буква/цифра) или конце строки. Иначе «Тип услуги Автосервис»
# ошибочно нашёлся бы внутри «Тип услуги Автосервис, аренда».
# Группировка по префиксу ускоряет поиск: движок проверяет длинную альтернацию
# только там, где уже совпал префикс, а не на каждой позиции строки.
def build_tag_regex(segments):
    by_prefix = {}
    for s in segments:
        for p in TAX_PREFIXES:
            if s.startswith(p + ' '):
                by_prefix.setdefault(p, []).append(s[len(p) + 1:])
    branches = [re.escape(p) + r' (?:' + '|'.join(re.escape(v) for v in vals) + ')'
                for p, vals in by_prefix.items()]
    return re.compile('(?:' + '|'.join(branches) + r')(?=\s+[A-ZА-ЯЁ0-9]|\s*$)')

TAG_RE = build_tag_regex(TAX_SEGMENTS)

def _tags_chunk(texts):
    return ['|'.join(sorted({m.group(0) for m in TAG_RE.finditer(t or '')})) for t in texts]

t = time.time()
items['tags'] = parallel_map(_tags_chunk, items['item_infm_params_text'].tolist())
print(f'теги объявлений ({time.time() - t:.0f} с), доля объявлений хотя бы с одним тегом: {(items.tags != "").mean():.2%}')
display(items[['item_infm_params_text', 'tags']].head(5))

теги объявлений (11 с), доля объявлений хотя бы с одним тегом: 94.97%


,item_infm_params_text,tags
0,Вид услуги Компьютерная помощь Место оказания услуг пр-т Ленина Тип стоимости за услугу Начальная цена График работы...,Вид услуги Компьютерная помощь|Тип услуги Компьютеры
1,"Вид услуги Обучение, курсы Место оказания услуг ул. Чернышевского, 35 Тип услуги Детское развитие, логопеды Тип стои...","Вид услуги Обучение, курсы|Тип услуги Детское развитие, логопеды"
2,"Вид услуги Красота, здоровье Место оказания услуг городской округ Иваново, Фрунзенский район Тип услуги Услуги парик...","Вид услуги Красота, здоровье|Тип услуги Услуги парикмахера"
3,"Вид услуги Строительство Место оказания услуг Севастопольский пр-т, 28к4 Тип стоимости за услугу Работа по договору ...",Вид услуги Строительство|Тип услуги Другое
4,"Вид услуги Обучение, курсы Место оказания услуг Волгоградская обл., Волжский, пл. имени В. И. Ленина Тип услуги Пред...","Вид услуги Обучение, курсы|Тип услуги Предметы школы и вуза"


### Проверка: выполняются ли фильтры запроса у выбранного объявления?

Если почти всегда (95%+), фильтр можно использовать как **жёсткий отбор кандидатов**, а не как мягкий признак.

In [15]:
RATING_RE = re.compile(r'Рейтинг пользователя (\d)')

def filter_satisfied(f_tax, f_other, tags, params, rating):
    """True, если объявление проходит все фильтры запроса."""
    if f_tax:
        tag_set = set(tags.split('|')) if tags else set()
        if not set(f_tax.split('|')) <= tag_set:
            return False
    if f_other:
        m = RATING_RE.search(f_other)
        if m and not (rating == rating and rating >= int(m.group(1))):   # rating == rating -> не NaN
            return False
        rest = RATING_RE.sub('', f_other)
        rest = re.sub(r'звезд[ыа]? и выше', '', rest).strip()
        if rest and rest not in (params or ''):
            return False
    return True

chk = (pairs[pairs.split.isin(['train', 'val'])]
       .merge(queries[['qid', 'f_tax', 'f_other']], on='qid')
       .merge(items[['item_id', 'tags', 'item_infm_params_text', 'item_rating']], on='item_id'))
chk = chk[(chk.f_tax != '') | (chk.f_other != '')]

tax_only = chk[chk.f_tax != '']
tax_ok = np.mean([set(f.split('|')) <= set(t.split('|')) for f, t in zip(tax_only.f_tax, tax_only.tags)])
all_ok = np.mean([filter_satisfied(*r) for r in chk[['f_tax', 'f_other', 'tags', 'item_infm_params_text', 'item_rating']].itertuples(index=False)])
print(f'пар с непустым фильтром: {len(chk):,}')
print(f'сегменты таксономии выполнены:  {tax_ok:.2%}')
print(f'все условия фильтра выполнены:  {all_ok:.2%}')
DIAG['фильтр: таксономия выполнена у выбранных'] = tax_ok
DIAG['фильтр: все условия выполнены у выбранных'] = all_ok

bad = chk[[not (set(f.split('|')) <= set(t.split('|'))) for f, t in zip(chk.f_tax, chk.tags)] & (chk.f_tax != '')]
print('\nпримеры НЕвыполненного фильтра (надо посмотреть глазами):')
display(bad[['f_tax', 'tags', 'item_infm_params_text']].head(8))
del chk, tax_only, bad; _ = gc.collect()

пар с непустым фильтром: 301,722
сегменты таксономии выполнены:  90.11%
все условия фильтра выполнены:  85.48%

примеры НЕвыполненного фильтра (надо посмотреть глазами):


,f_tax,tags,item_infm_params_text
0,"Вид услуги Сад, благоустройство",Вид услуги Строительство|Тип услуги Фундаментные и бетонные работы Чем вы занимаетесь Бетонирование конструкций,"Вид услуги Строительство Место оказания услуг Амурская область, Благовещенск, улица Ленина, 72 Тип стоимости за услу..."
5,Вид услуги Монтаж и установка техники,"Вид услуги Ремонт и обслуживание техники|Тип услуги Варочные панели, духовые шкафы","Вид услуги Ремонт и обслуживание техники Тип услуги Варочные панели, духовые шкафы Место оказания услуг Амурская обл..."
11,"Тип услуги автосервиса Помощь на дороге|Вид услуги Автосервис, аренда|Тип услуги Автосервис","Вид услуги Автосервис, аренда|Тип услуги Автосервис|Тип услуги автосервиса Диагностика и ремонт авто Можно со своими...","Вид услуги Автосервис, аренда Тип услуги Автосервис Место оказания услуг Владимирская область, Петушки Гарантия Опыт..."
15,"Тип услуги автосервиса Кузовной ремонт и покраска|Вид услуги Автосервис, аренда|Тип услуги Автосервис","Вид услуги Автосервис, аренда|Тип услуги Автосервис|Тип услуги автосервиса Кузовной ремонт и покраска Можно со своим...","Вид услуги Автосервис, аренда Тип услуги Автосервис Место оказания услуг Пенза, Светлая улица, 2Б Гарантия Опыт рабо..."
51,"Тип услуги автосервиса Диагностика и ремонт авто|Вид услуги Автосервис, аренда|Тип услуги Автосервис","Вид услуги Автосервис, аренда|Тип услуги Автосервис|Тип услуги автосервиса Диагностика и ремонт авто Можно со своими...","Вид услуги Автосервис, аренда Тип услуги Автосервис Место оказания услуг Ростов-на-Дону, Первомайский район Гарантия..."
52,"Тип услуги автосервиса Диагностика и ремонт авто|Вид услуги Автосервис, аренда|Тип услуги Автосервис","Вид услуги Автосервис, аренда|Тип услуги Автосервис|Тип услуги автосервиса Диагностика и ремонт авто Можно со своими...","Вид услуги Автосервис, аренда Тип услуги Автосервис Место оказания услуг Ростов-на-Дону, посёлок Чкаловский, Сальски..."
53,"Тип услуги автосервиса Диагностика и ремонт авто|Вид услуги Автосервис, аренда|Тип услуги Автосервис","Вид услуги Автосервис, аренда|Тип услуги Автосервис|Тип услуги автосервиса Диагностика и ремонт авто Можно со своими...","Вид услуги Автосервис, аренда Тип услуги Автосервис Место оказания услуг Ростов-на-Дону, Нахичевань, улица Сарьяна, ..."
54,"Тип услуги автосервиса Диагностика и ремонт авто|Вид услуги Автосервис, аренда|Тип услуги Автосервис","Вид услуги Автосервис, аренда|Тип услуги Автосервис|Тип услуги автосервиса Диагностика и ремонт авто Можно со своими...","Вид услуги Автосервис, аренда Тип услуги Автосервис Место оказания услуг Профсоюзная ул., 131 Гарантия Опыт работы 5..."


## 10. Гео: где находится пользователь

**Про удаление координат.** Координат запроса у нас действительно нет, но их можно **оценить**, так что удалять координаты объявлений не стоит.

- **Центроид по объявлениям**: медиана координат объявлений с тем же `location_id`. Подходит для городов, где есть объявления.
- **Центроид по выборам в train**: медиана координат объявлений, которые выбирали люди, искавшие из этой локации.
  Это закрывает главную проблему из EDA: у **17% запросов бенчмарка в их `location_id` нет ни одного объявления**.
  Скорее всего, это «региональные» id (область, край), а объявления привязаны к городам. Жёсткий фильтр по `location_id`
  дал бы на этих запросах Recall = 0, а фильтр по радиусу от оценённого центра работает.

Центроиды по выборам считаем только на `split == 'train'`, чтобы валидация оставалась честной.

In [16]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = (np.radians(np.asarray(x, dtype='float64')) for x in (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

# (а) центроиды по объявлениям с этим location_id
loc_items = (items.groupby('item_location_id')
             .agg(lat_items=('lat', 'median'), lon_items=('lon', 'median'), n_items=('item_id', 'size')))

# (б) центроиды по выборам в train
ch = (pairs[pairs.split == 'train'][['qid', 'item_id']]
      .merge(queries[['qid', 'search_location_id']], on='qid')
      .merge(items[['item_id', 'lat', 'lon']], on='item_id'))
loc_choice = (ch.groupby('search_location_id')
              .agg(lat_choice=('lat', 'median'), lon_choice=('lon', 'median'), n_choices=('item_id', 'size')))

locs = loc_items.join(loc_choice, how='outer')
locs.index.name = 'location_id'

# Оценка «где пользователь»: если у локации достаточно своих объявлений, берём их центроид (точнее всего),
# иначе центроид выборов (региональные id), иначе то, что есть.
use_items = locs.n_items.fillna(0) >= 3
use_choice = ~use_items & (locs.n_choices.fillna(0) >= 3)
locs['lat'] = np.where(use_items, locs.lat_items, np.where(use_choice, locs.lat_choice, locs.lat_items.fillna(locs.lat_choice)))
locs['lon'] = np.where(use_items, locs.lon_items, np.where(use_choice, locs.lon_choice, locs.lon_items.fillna(locs.lon_choice)))
locs['geo_source'] = np.select([use_items, use_choice], ['items', 'choices'], default='fallback')
locs.loc[locs.lat.isna(), 'geo_source'] = 'unknown'
print(locs.geo_source.value_counts())

queries = queries.merge(locs[['lat', 'lon']].rename(columns={'lat': 'loc_lat', 'lon': 'loc_lon'}),
                        left_on='search_location_id', right_index=True, how='left')
queries['loc_n_items'] = queries.search_location_id.map(locs.n_items).fillna(0).astype('int32')

b = queries.split == 'bench'
print(f'\nбенчмарк: оценка координат есть у {queries.loc[b, "loc_lat"].notna().mean():.2%} запросов')
DIAG['бенчмарк: есть оценка координат'] = queries.loc[b, 'loc_lat'].notna().mean()
del ch; _ = gc.collect()

geo_source
items       2336
fallback     960
choices      199
Name: count, dtype: int64

бенчмарк: оценка координат есть у 100.00% запросов


### Какой гео-фильтр безопасен: проверка на валидации

Для пар val (объявление есть в корпусе) считаем, какая доля выбранных объявлений проходит каждый вариант гео-фильтра.
Это **потолок Recall** при таком фильтре. Затем для бенчмарка смотрим, сколько кандидатов оставляет каждый вариант.

In [17]:
v = (pairs[(pairs.split == 'val') & pairs.in_corpus][['qid', 'item_id']]
     .merge(queries[['qid', 'search_location_id', 'loc_lat', 'loc_lon']], on='qid')
     .merge(items[['item_id', 'item_location_id', 'lat', 'lon', 'is_remote_text']], on='item_id'))
v['dist'] = haversine_km(v.loc_lat, v.loc_lon, v.lat, v.lon)
v['same_id'] = v.search_location_id == v.item_location_id

rows = [{'фильтр': 'тот же location_id', 'проходит выбранных': v.same_id.mean()}]
for R in [10, 30, 50, 100, 300]:
    rows.append({'фильтр': f'тот же id ИЛИ <= {R} км', 'проходит выбранных': (v.same_id | (v.dist <= R)).mean()})
rows.append({'фильтр': 'тот же id ИЛИ <= 50 км ИЛИ удалёнка', 'проходит выбранных': (v.same_id | (v.dist <= 50) | (v.is_remote_text == 1)).mean()})
geo_tbl = pd.DataFrame(rows).set_index('фильтр')
display(geo_tbl.style.format('{:.2%}'))
DIAG['val: тот же location_id'] = v.same_id.mean()
DIAG['val: тот же id или <= 50 км'] = (v.same_id | (v.dist <= 50)).mean()

far = v[~v.same_id & (v.dist > 300)]
print(f'\nдальние выборы (>300 км): {len(far) / len(v):.2%} пар; из них с признаком удалёнки: {far.is_remote_text.mean():.2%} '
      f'(в целом по корпусу: {items.loc[items.in_corpus, "is_remote_text"].mean():.2%})')

,проходит выбранных
фильтр,
тот же location_id,77.29%
тот же id ИЛИ <= 10 км,86.38%
тот же id ИЛИ <= 30 км,93.17%
тот же id ИЛИ <= 50 км,94.42%
тот же id ИЛИ <= 100 км,95.70%
тот же id ИЛИ <= 300 км,97.21%
тот же id ИЛИ <= 50 км ИЛИ удалёнка,97.11%



дальние выборы (>300 км): 2.79% пар; из них с признаком удалёнки: 52.94% (в целом по корпусу: 36.89%)


In [18]:
# Сколько кандидатов оставляет гео-фильтр для запросов бенчмарка (считаем по уникальным локациям)
corp = items[items.in_corpus].reset_index(drop=True)
c_lat, c_lon, c_loc = corp.lat.values, corp.lon.values, corp.item_location_id.values
bq = queries[queries.split == 'bench']

pool = {}
for loc_id, g in bq.groupby('search_location_id'):
    lat0, lon0 = g.loc_lat.iloc[0], g.loc_lon.iloc[0]
    same = c_loc == loc_id
    if np.isnan(lat0):
        pool[loc_id] = {'same_id': same.sum(), 'id|30km': same.sum(), 'id|50km': same.sum(), 'id|100km': same.sum()}
        continue
    d = haversine_km(lat0, lon0, c_lat, c_lon)
    pool[loc_id] = {'same_id': same.sum(), 'id|30km': (same | (d <= 30)).sum(),
                    'id|50km': (same | (d <= 50)).sum(), 'id|100km': (same | (d <= 100)).sum()}
pool = pd.DataFrame.from_dict(pool, orient='index')
bench_pool = bq[['search_location_id']].join(pool, on='search_location_id').drop(columns='search_location_id')
print('Размер пула кандидатов у запросов бенчмарка:')
display(bench_pool.describe(percentiles=[.1, .5, .9]).T[['10%', '50%', '90%', 'max']])
print('доля запросов с пустым пулом:')
display((bench_pool == 0).mean().to_frame('share_empty').T.style.format('{:.2%}'))
DIAG['бенчмарк: пустой пул по id'] = (bench_pool['same_id'] == 0).mean()
DIAG['бенчмарк: пустой пул по id|50км'] = (bench_pool['id|50km'] == 0).mean()
del corp, v, far; _ = gc.collect()

Размер пула кандидатов у запросов бенчмарка:


,10%,50%,90%,max
same_id,0.0,1638.0,19543.0,19543.0
id|30km,658.0,3273.0,27393.0,27434.0
id|50km,722.0,3503.0,32738.0,33223.0
id|100km,1120.0,4057.0,39239.0,39718.0


доля запросов с пустым пулом:


,same_id,id|30km,id|50km,id|100km
share_empty,17.41%,0.00%,0.00%,0.00%


## 11. Категории и микрокатегории: проверка перед фильтрацией

- `search_category` почти всегда 114, и выбранное объявление почти всегда той же категории. Смотрим, сколько корпуса отсекает фильтр по категории
- в корпусе 752 микрокатегории, а в train всего 212. Если объявления вне этих 212 почти не выбирают, их можно сразу исключить

In [19]:
bq = queries[queries.split == 'bench']
print('search_category в бенчмарке:', bq.search_category.value_counts().to_dict())
corp_cat = items.loc[items.in_corpus, 'item_category_id'].value_counts(normalize=True)
print('\nкатегории корпуса (топ-8):'); display(corp_cat.head(8).to_frame('share'))

v = pairs[(pairs.split == 'val') & pairs.in_corpus].merge(queries[['qid', 'search_category']], on='qid') \
     .merge(items[['item_id', 'item_category_id', 'item_microcat_id']], on='item_id')
cat_ok = (v.search_category == v.item_category_id).mean()
print(f'val: категория выбранного объявления == search_category: {cat_ok:.2%}')
DIAG['val: категория совпадает'] = cat_ok

train_mc = set(pairs[pairs.split == 'train'].merge(items[['item_id', 'item_microcat_id']], on='item_id').item_microcat_id)
corp_mc_ok = items.loc[items.in_corpus, 'item_microcat_id'].isin(train_mc).mean()
val_mc_ok = v.item_microcat_id.isin(train_mc).mean()
print(f'доля корпуса в микрокатегориях, встречавшихся в train: {corp_mc_ok:.2%}')
print(f'val: выбранное объявление в «train-микрокатегории»:    {val_mc_ok:.2%}')
DIAG['корпус в train-микрокатегориях'] = corp_mc_ok
DIAG['val: выбранное в train-микрокатегории'] = val_mc_ok
del v; _ = gc.collect()

search_category в бенчмарке: {114: 2230, 0: 222}

категории корпуса (топ-8):


,share
item_category_id,
114,0.990085
19,0.001887
112,0.000745
40,0.000645
33,0.000544
10,0.000518
20,0.000444
25,0.000433


val: категория выбранного объявления == search_category: 99.97%
доля корпуса в микрокатегориях, встречавшихся в train: 99.07%
val: выбранное объявление в «train-микрокатегории»:    100.00%


## 12. Сохранение

- `items.parquet`: объявления с чистыми числовыми признаками, `*_norm` (для нейросетей), `*_lem` (для BM25), `tags`, `is_remote_text`, координаты
- `queries.parquet`: запросы train/val/bench с `query_norm`, `query_lem`, сегментами фильтров, оценкой координат
- `pairs.parquet`: запрос → выбранное объявление, со `split` и флагом `in_corpus`
- `locations.parquet`: таблица локаций с центроидами

Сырые тексты оставляем: для отладки полезно видеть оригинал.

In [20]:
items.to_parquet(f'{OUT_DIR}/items.parquet', index=False)
queries.to_parquet(f'{OUT_DIR}/queries.parquet', index=False)
pairs.to_parquet(f'{OUT_DIR}/pairs.parquet', index=False)
locs.reset_index().to_parquet(f'{OUT_DIR}/locations.parquet', index=False)
for f in sorted(os.listdir(OUT_DIR)):
    print(f'{f:<22} {os.path.getsize(os.path.join(OUT_DIR, f)) / 2**20:8.1f} МБ')

items.parquet            2143.9 МБ
locations.parquet           0.1 МБ
pairs.parquet              13.0 МБ
queries.parquet            31.1 МБ


## 13. Итоговая диагностика

In [21]:
def fmt(v):
    if isinstance(v, (int, np.integer)):
        return f'{v:,}'
    return f'{v:.2%}'
display(pd.Series({k: fmt(v) for k, v in DIAG.items()}, name='value').to_frame())

,value
val: запросов,"3,000"
фильтр: таксономия выполнена у выбранных,90.11%
фильтр: все условия выполнены у выбранных,85.48%
бенчмарк: есть оценка координат,100.00%
val: тот же location_id,77.29%
val: тот же id или <= 50 км,94.42%
бенчмарк: пустой пул по id,17.41%
бенчмарк: пустой пул по id|50км,0.00%
val: категория совпадает,99.97%
корпус в train-микрокатегориях,99.07%


In [22]:
OUT_DIR = '/kaggle/working/prep'
q  = pd.read_parquet(f'{OUT_DIR}/queries.parquet')
p  = pd.read_parquet(f'{OUT_DIR}/pairs.parquet')
it = pd.read_parquet(f'{OUT_DIR}/items.parquet',
                     columns=['item_id', 'item_category_id', 'in_corpus', 'title_norm'])

# 1) какие категории объявлений выбирали в train при search_category == 0
c0 = p.merge(q[['qid', 'search_category']], on='qid').query('search_category == 0')
print(c0.merge(it, on='item_id').item_category_id.value_counts())

# 2) как выглядят запросы бенчмарка с категорией 0
print(q.query("split == 'bench' and search_category == 0").search_query.sample(20, random_state=0).tolist())

# 3) что за объявления корпуса вне категории 114
print(it[it.in_corpus & (it.item_category_id != 114)].title_norm.sample(20, random_state=0).tolist())

item_category_id
114    34
Name: count, dtype: int64
['курсы украшения', 'обложка меню', 'база', 'специалист по настройкам окон пластиковых', 'микроволновая печь ремонт', 'вакансия репетитор онлайн математика информатика', '223 фз юрист', 'творческая студия для подростков', 'порошковая покраска высокая камера', 'водитель с авто', 'снять базу отдыха', 'карта т-банк 1500', 'стеклянный шар с наклейкой', 'репетитор английский', 'баня индивидуальное парение', 'ключ пожарный', 'трудоустройство яндекс такси', 'разработчик проектировщик мебели', 'тренер по восстановлению', 'тренер по силовым тренировкам']
['Забор из металлического штакетника', 'Шкаф на балкон на заказ', 'Яндекс Директ, настройка рекламы', 'Офис в центре Нижнего Новгорода 181.3 м²', 'Нержавеющий реактор, диссольвер 250 л', 'Газорезчик, маляр, стропальщик', 'Жби кольца,выгребные ямы,септики под ключ', 'Тротуарная плитка и бордюри', 'Квартира-студия, 45 м², 4 кровати', 'Инженер-конструктор', 'Таунхаус 140 м² на участке 2,5 сот.',

In [23]:
import os, re, gc, json, time, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 250)
pd.set_option('display.max_colwidth', 60)

PREP_DIR = '/kaggle/working/prep' if os.path.isdir('/kaggle/working/prep') else './prep'
FEAT_DIR = PREP_DIR.replace('prep', 'features')
os.makedirs(FEAT_DIR, exist_ok=True)

SEED = 42
GEO_RADIUS_KM = 50       # гео-пул: тот же id ИЛИ ближе этого радиуса ИЛИ удалёнка
K_POOL = 200             # кандидатов из гео-пула
K_GLOBAL = 50            # кандидатов по всему корпусу без гео (дальние выборы)
MAX_TRAIN_Q = 15_000     # сколько train-запросов берём для CatBoost (хватит с запасом)
BATCH = 128              # запросов за один проход BM25
BM25_K1, BM25_B = 1.2, 0.75
TITLE_WEIGHT = 2         # в общем поле заголовок учитывается дважды: он точнее описания

try:
    import psutil
    def mem(tag=''):
        print(f'[mem] {tag}: {psutil.Process().memory_info().rss / 2**30:.1f} ГБ')
except ImportError:
    def mem(tag=''): pass
print(PREP_DIR, '->', FEAT_DIR)

/kaggle/working/prep -> /kaggle/working/features


In [24]:
ITEM_COLS = ['item_id', 'item_category_id', 'item_microcat_id', 'item_location_id', 'lat', 'lon', 'is_remote_text',
             'item_rating', 'rating_isna', 'reviews_log', 'price_log', 'price_is_symbolic',
             'item_is_phone_hidden', 'item_is_message_forbidden',
             'title_norm', 'params_norm', 'title_lem', 'params_lem', 'desc_lem']
items = pd.read_parquet(f'{PREP_DIR}/items.parquet', columns=ITEM_COLS,
                        filters=[('in_corpus', '==', True)]).reset_index(drop=True)
for c in ['title_lem', 'params_lem', 'desc_lem', 'params_norm', 'title_norm']:
    items[c] = items[c].fillna('')

queries = pd.read_parquet(f'{PREP_DIR}/queries.parquet',
                          columns=['qid', 'split', 'search_query', 'query_norm', 'query_lem', 'search_location_id',
                                   'search_category', 'search_infm_params_text', 'loc_lat', 'loc_lon', 'loc_n_items'])
pairs = pd.read_parquet(f'{PREP_DIR}/pairs.parquet')
print(f'корпус: {len(items):,} | запросов: {len(queries):,} | пар: {len(pairs):,}')
mem('после загрузки')

корпус: 189,212 | запросов: 356,914 | пар: 467,054
[mem] после загрузки: 20.8 ГБ


In [25]:
id2idx = pd.Series(np.arange(len(items)), index=items['item_id'])
pos_pairs = pairs[pairs.in_corpus & pairs.item_id.isin(id2idx.index)]
POS = pos_pairs.groupby('qid')['item_id'].apply(lambda s: set(id2idx[s].values)).to_dict()   # qid -> {индексы строк}

val_q = queries[(queries.split == 'val') & queries.qid.isin(POS)].reset_index(drop=True)
tr_all = queries[(queries.split == 'train') & queries.qid.isin(POS)]
tr_q = tr_all.sample(n=min(MAX_TRAIN_Q, len(tr_all)), random_state=SEED).reset_index(drop=True)
print(f'train-запросов с объявлением в корпусе: {len(tr_all):,}, берём {len(tr_q):,}')
print(f'val-запросов: {len(val_q):,}')
print(f'релевантных на запрос: train {np.mean([len(POS[q]) for q in tr_q.qid]):.2f}, val {np.mean([len(POS[q]) for q in val_q.qid]):.2f}')

train-запросов с объявлением в корпусе: 23,363, берём 15,000
val-запросов: 3,000
релевантных на запрос: train 1.12, val 1.02


In [26]:
def bm25_matrix(tf, k1=BM25_K1, b=BM25_B):
    """Матрица tf (объявления × слова) -> матрица весов BM25 той же формы (CSR)."""
    tf = tf.tocsr().astype(np.float32)
    n_docs = tf.shape[0]
    dl = np.asarray(tf.sum(axis=1)).ravel()
    avgdl = max(dl.mean(), 1e-9)
    df = np.bincount(tf.indices, minlength=tf.shape[1])
    idf = np.log(1 + (n_docs - df + 0.5) / (df + 0.5)).astype(np.float32)
    rows = np.repeat(np.arange(n_docs), np.diff(tf.indptr))
    data = tf.data * (k1 + 1) / (tf.data + k1 * (1 - b + b * dl[rows] / avgdl)) * idf[tf.indices]
    return sp.csr_matrix((data.astype(np.float32), tf.indices, tf.indptr), shape=tf.shape)

t = time.time()
cv = CountVectorizer(token_pattern=r'\S+', lowercase=False, dtype=np.float32)
n = len(items)
# один проход токенизации по трём полям сразу, потом режем по строкам
X = cv.fit_transform(pd.concat([items.title_lem, items.params_lem, items.desc_lem], ignore_index=True)).tocsr()
TF = {'title': X[:n], 'params': X[n:2 * n], 'desc': X[2 * n:]}
del X
TF['all'] = (TITLE_WEIGHT * TF['title'] + TF['params'] + TF['desc']).tocsr()

W = {f: bm25_matrix(TF[f]) for f in ['all', 'title', 'params', 'desc']}
# транспонированная матрица для быстрого скоринга батча: (запросы × слова) @ (слова × объявления)
# читает только строки слов, которые есть в запросах, а не всю матрицу
WT_ALL = W['all'].T.tocsr()
BIN = {f: (TF[f] > 0).astype(np.float32).tocsr() for f in ['title', 'all']}   # «есть ли слово в поле», для покрытия

item_len = {f'{f}_len': np.asarray(TF[f].sum(axis=1)).ravel().astype(np.float32) for f in ['title', 'params', 'desc']}
del TF; gc.collect()
print(f'словарь: {len(cv.vocabulary_):,} слов | {time.time() - t:.0f} с')
mem('после BM25-матриц')

словарь: 399,994 слов | 49 с
[mem] после BM25-матриц: 21.6 ГБ


In [27]:
URL_RE   = re.compile(r'https?://\S+|www\.\S+')
EMAIL_RE = re.compile(r'\S+@\S+\.\S+')
PHONE_RE = re.compile(r'(?:\+7|\b8)[\s\-(]*\d{3}[\s\-)]*\d{3}[\s\-]*\d{2}[\s\-]*\d{2}')
NOISE_RE = re.compile(r'[^\w\s.,:;!?()«»"\'%№/+\-]')
WS_RE    = re.compile(r'\s+')
RATING_RE = re.compile(r'Рейтинг пользователя (\d)\S*\s+звезд\S*\s+и\s+выше')

def normalize_for_model(s):
    if not isinstance(s, str) or not s:
        return ''
    s = s.replace('\\n', ' ').replace('ё', 'е').replace('Ё', 'Е')
    s = URL_RE.sub(' ', s); s = EMAIL_RE.sub(' ', s); s = PHONE_RE.sub(' ', s); s = NOISE_RE.sub(' ', s)
    return WS_RE.sub(' ', s).strip()

def parse_filter(f):
    """-> (минимальный рейтинг или NaN, текстовая часть фильтра)."""
    f = normalize_for_model(f)
    m = RATING_RE.search(f)
    return (float(m.group(1)) if m else np.nan), WS_RE.sub(' ', RATING_RE.sub(' ', f)).strip()

def covered(f, params):
    """Можно ли разрезать фильтр на куски подряд идущих слов, каждый из которых есть в params."""
    w = f.split(' ')
    i, n_ = 0, len(w)
    while i < n_:
        j = i
        while j < n_ and ' '.join(w[i:j + 1]) in params:
            j += 1
        if j == i:
            return False
        i = j
    return True

print(parse_filter('Рейтинг пользователя 4 звезды и выше Вид услуги Красота, здоровье'))

(4.0, 'Вид услуги Красота, здоровье')


In [28]:
I_LAT = np.radians(items.lat.values.astype('float64'))
I_LON = np.radians(items.lon.values.astype('float64'))
I_LOC = items.item_location_id.values
I_REMOTE = items.is_remote_text.values.astype(bool)
CAT_MASK = {114: items.item_category_id.values == 114}
ALL_TRUE = np.ones(len(items), dtype=bool)
I_MC = items.item_microcat_id.values
I_PARAMS = items.params_norm.values
I_RATING = items.item_rating.values.astype('float64')

def dist_km(lat, lon):
    """Расстояние от точки (градусы) до всех объявлений корпуса, км."""
    lat, lon = np.radians(lat), np.radians(lon)
    a = np.sin((I_LAT - lat) / 2) ** 2 + np.cos(lat) * np.cos(I_LAT) * np.sin((I_LON - lon) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def top_k(idx, scores, k):
    """k индексов из idx с наибольшим скором, отсортированные по убыванию."""
    if len(idx) == 0 or k <= 0:
        return idx[:0]
    k = min(k, len(idx))
    part = idx[np.argpartition(-scores[idx], k - 1)[:k]]
    return part[np.argsort(-scores[part], kind='stable')]

In [29]:
def build_features(qdf, name):
    t0 = time.time()
    Q = cv.transform(qdf['query_lem'].fillna('')).tocsr()
    Q.data[:] = 1                                    # важно наличие слова в запросе, а не частота
    q_n_terms = np.asarray(Q.sum(axis=1)).ravel()
    filters = [parse_filter(f) for f in qdf['search_infm_params_text'].fillna('')]
    cols = {k: [] for k in ['qid_idx', 'cand', 'bm25_all', 'bm25_title', 'bm25_params', 'bm25_desc', 'bm25_rel',
                            'rank_pool', 'rank_global', 'in_pool', 'cov_title', 'cov_all', 'mc_share_top20',
                            'dist_km', 'same_loc', 'filter_text_ok', 'rating_ok', 'filter_ok', 'pool_size', 'target']}
    stats = {'cand_recall': [], 'bm25_pool_recall50': []}

    for b0 in range(0, len(qdf), BATCH):
        qb = Q[b0:b0 + BATCH]
        S = (qb @ WT_ALL).toarray()                           # (запросы батча × корпус): BM25 по полю all
        for j in range(qb.shape[0]):
            qi = b0 + j
            row = qdf.iloc[qi]
            s = S[j]
            cat = CAT_MASK.get(int(row.search_category), ALL_TRUE)

            # --- гео-пул
            if pd.notna(row.loc_lat):
                d = dist_km(row.loc_lat, row.loc_lon)
                geo = (I_LOC == row.search_location_id) | (d <= GEO_RADIUS_KM)
            else:
                d = np.full(len(items), np.nan)
                geo = ALL_TRUE
            pool_mask = cat & geo
            if not pool_mask.any():
                pool_mask = cat
            pool_idx = np.flatnonzero(pool_mask)

            # --- кандидаты из двух источников
            top_pool = top_k(pool_idx, s, K_POOL)
            top_glob = top_k(np.flatnonzero(cat), s, K_GLOBAL)
            cand = np.unique(np.concatenate([top_pool, top_glob]))
            rank_pool = pd.Series(np.arange(len(top_pool)), index=top_pool).reindex(cand).fillna(K_POOL).values
            rank_glob = pd.Series(np.arange(len(top_glob)), index=top_glob).reindex(cand).fillna(K_GLOBAL).values

            # --- скоры по полям и покрытие слов запроса (только для кандидатов)
            qv = qb[j].T
            f_title  = np.asarray((W['title'][cand] @ qv).todense()).ravel()
            f_params = np.asarray((W['params'][cand] @ qv).todense()).ravel()
            f_desc   = np.asarray((W['desc'][cand] @ qv).todense()).ravel()
            nt = max(q_n_terms[qi], 1)
            c_title = np.asarray((BIN['title'][cand] @ qv).todense()).ravel() / nt
            c_all   = np.asarray((BIN['all'][cand] @ qv).todense()).ravel() / nt

            # --- «о чём запрос»: распределение микрокатегорий в топ-20 пула
            mc_top = pd.Series(I_MC[top_pool[:20]]).value_counts(normalize=True)
            mc_share = pd.Series(I_MC[cand]).map(mc_top).fillna(0).values

            # --- фильтр
            min_r, rest = filters[qi]
            ftxt = np.array([covered(rest, I_PARAMS[c]) for c in cand]) if rest else np.ones(len(cand), bool)
            r_ok = (I_RATING[cand] >= min_r) if min_r == min_r else np.ones(len(cand), bool)

            best = s[top_pool[0]] if len(top_pool) else 0.0
            pos = POS[row.qid]
            tgt = np.array([c in pos for c in cand])

            cols['qid_idx'].append(np.full(len(cand), qi)); cols['cand'].append(cand)
            cols['bm25_all'].append(s[cand]); cols['bm25_title'].append(f_title)
            cols['bm25_params'].append(f_params); cols['bm25_desc'].append(f_desc)
            cols['bm25_rel'].append(s[cand] / best if best > 0 else np.zeros(len(cand)))
            cols['rank_pool'].append(rank_pool); cols['rank_global'].append(rank_glob)
            cols['in_pool'].append(pool_mask[cand]); cols['cov_title'].append(c_title); cols['cov_all'].append(c_all)
            cols['mc_share_top20'].append(mc_share)
            cols['dist_km'].append(d[cand]); cols['same_loc'].append(I_LOC[cand] == row.search_location_id)
            cols['filter_text_ok'].append(ftxt); cols['rating_ok'].append(r_ok); cols['filter_ok'].append(ftxt & r_ok)
            cols['pool_size'].append(np.full(len(cand), len(pool_idx))); cols['target'].append(tgt)

            stats['cand_recall'].append(tgt.sum() / len(pos))
            stats['bm25_pool_recall50'].append(len(set(top_pool[:50]) & pos) / len(pos))

        if (b0 // BATCH) % 20 == 0:
            print(f'  {name}: {min(b0 + BATCH, len(qdf)):,}/{len(qdf):,} запросов, {time.time() - t0:.0f} с')

    df = pd.DataFrame({k: np.concatenate(v) for k, v in cols.items()})
    qi, ci = df.pop('qid_idx').values, df.pop('cand').values

    # --- служебные колонки
    df.insert(0, 'qid', qdf['qid'].values[qi])
    df.insert(1, 'item_id', items['item_id'].values[ci])
    df['n_pos_total'] = qdf['qid'].map(lambda q: len(POS[q])).values[qi]

    # --- признаки запроса
    df['q_n_terms'] = q_n_terms[qi]
    df['q_len_words'] = qdf['query_norm'].fillna('').str.split().str.len().values[qi]
    df['search_category'] = qdf['search_category'].values[qi]
    df['loc_n_items'] = qdf['loc_n_items'].values[qi]
    df['geo_known'] = qdf['loc_lat'].notna().values[qi]
    df['has_filter'] = np.array([bool(r) or m == m for m, r in filters])[qi]

    # --- признаки объявления
    for c in ['item_microcat_id', 'is_remote_text', 'item_rating', 'rating_isna', 'reviews_log', 'price_log',
              'price_is_symbolic', 'item_is_phone_hidden', 'item_is_message_forbidden']:
        df[c] = items[c].values[ci]
    for c, v in item_len.items():
        df[c] = v[ci]

    # --- компактные типы
    for c in df.columns:
        if df[c].dtype == bool:
            df[c] = df[c].astype('int8')
        elif df[c].dtype == np.float64:
            df[c] = df[c].astype('float32')
    df['target'] = df['target'].astype('int8')

    print(f'{name}: {len(qdf):,} запросов, {len(df):,} строк, {time.time() - t0:.0f} с')
    return df, {k: float(np.mean(v)) for k, v in stats.items()}

val_df, val_stats = build_features(val_q, 'val')
mem('после val')

  val: 128/3,000 запросов, 3 с
  val: 2,688/3,000 запросов, 63 с
val: 3,000 запросов, 721,220 строк, 71 с
[mem] после val: 21.8 ГБ


In [30]:
train_df, train_stats = build_features(tr_q, 'train')
mem('после train')

  train: 128/15,000 запросов, 3 с
  train: 2,688/15,000 запросов, 63 с
  train: 5,248/15,000 запросов, 125 с
  train: 7,808/15,000 запросов, 184 с
  train: 10,368/15,000 запросов, 244 с
  train: 12,928/15,000 запросов, 303 с
train: 15,000 запросов, 3,625,728 строк, 353 с
[mem] после train: 22.5 ГБ


In [31]:
SERVICE_COLS = ['qid', 'item_id', 'target', 'n_pos_total']
CAT_FEATURES = ['item_microcat_id', 'search_category']
FEATURES = [c for c in train_df.columns if c not in SERVICE_COLS]

assert list(train_df.columns) == list(val_df.columns), 'наборы колонок train и val различаются!'
train_df.to_parquet(f'{FEAT_DIR}/train.parquet', index=False)
val_df.to_parquet(f'{FEAT_DIR}/val.parquet', index=False)
with open(f'{FEAT_DIR}/features.json', 'w') as f:
    json.dump({'features': FEATURES, 'cat_features': CAT_FEATURES, 'service': SERVICE_COLS,
               'params': {'GEO_RADIUS_KM': GEO_RADIUS_KM, 'K_POOL': K_POOL, 'K_GLOBAL': K_GLOBAL}}, f, ensure_ascii=False, indent=2)
print(f'признаков: {len(FEATURES)} (категориальные: {CAT_FEATURES})')
print(FEATURES)
for f_ in sorted(os.listdir(FEAT_DIR)):
    print(f'{f_:<16} {os.path.getsize(os.path.join(FEAT_DIR, f_)) / 2**20:7.1f} МБ')

признаков: 35 (категориальные: ['item_microcat_id', 'search_category'])
['bm25_all', 'bm25_title', 'bm25_params', 'bm25_desc', 'bm25_rel', 'rank_pool', 'rank_global', 'in_pool', 'cov_title', 'cov_all', 'mc_share_top20', 'dist_km', 'same_loc', 'filter_text_ok', 'rating_ok', 'filter_ok', 'pool_size', 'q_n_terms', 'q_len_words', 'search_category', 'loc_n_items', 'geo_known', 'has_filter', 'item_microcat_id', 'is_remote_text', 'item_rating', 'rating_isna', 'reviews_log', 'price_log', 'price_is_symbolic', 'item_is_phone_hidden', 'item_is_message_forbidden', 'title_len', 'params_len', 'desc_len']
features.json        0.0 МБ
train.parquet      157.6 МБ
val.parquet         32.9 МБ


In [32]:
summary = pd.DataFrame({
    'train': [train_df.qid.nunique(), len(train_df), len(train_df) / train_df.qid.nunique(), train_df.target.mean(),
              train_stats['cand_recall'], train_stats['bm25_pool_recall50']],
    'val':   [val_df.qid.nunique(), len(val_df), len(val_df) / val_df.qid.nunique(), val_df.target.mean(),
              val_stats['cand_recall'], val_stats['bm25_pool_recall50']],
}, index=['запросов', 'строк', 'кандидатов на запрос', 'доля target=1',
          'потолок Recall (релевантное в кандидатах)', 'Recall@50: BM25 в пуле (бейзлайн)'])
display(summary.style.format('{:,.4f}'))

,train,val
запросов,"15,000.0000","3,000.0000"
строк,"3,625,728.0000","721,220.0000"
кандидатов на запрос,241.7152,240.4067
доля target=1,0.0044,0.0039
потолок Recall (релевантное в кандидатах),0.9495,0.9322
Recall@50: BM25 в пуле (бейзлайн),0.8709,0.8339


In [33]:
print('train: первые 10 строк')
display(train_df.head(10))
print('val: первые 10 строк')
display(val_df.head(10))

train: первые 10 строк


,qid,item_id,bm25_all,bm25_title,bm25_params,bm25_desc,bm25_rel,rank_pool,rank_global,in_pool,cov_title,cov_all,mc_share_top20,dist_km,same_loc,filter_text_ok,rating_ok,filter_ok,pool_size,target,n_pos_total,q_n_terms,q_len_words,search_category,loc_n_items,geo_known,has_filter,item_microcat_id,is_remote_text,item_rating,rating_isna,reviews_log,price_log,price_is_symbolic,item_is_phone_hidden,item_is_message_forbidden,title_len,params_len,desc_len
0,tr_caceaad574bb3bc4,cbc5d40a256e7685,2.579009,2.090788,2.303012,3.275523,0.172206,69.0,50.0,1,0.5,0.5,0.00,7.335962,1,1,1,1,3851,0,1,2.0,2,114,9807,1,0,84438,0,5.000000,0,5.187386,6.216606,0,0,0,5.0,162.0,138.0
1,tr_caceaad574bb3bc4,4cd525d3f3a7d82b,13.415874,0.000000,0.000000,12.205449,0.895807,200.0,23.0,0,0.0,0.5,0.00,726.337524,0,1,1,1,3851,0,1,2.0,2,114,9807,1,0,1289819,0,NaN,1,1.098612,8.294300,0,0,0,4.0,22.0,223.0
2,tr_caceaad574bb3bc4,c807cf48af140e98,2.536615,2.304721,1.864747,3.136667,0.169375,104.0,50.0,1,0.5,0.5,0.05,37.871632,0,1,1,1,3851,0,1,2.0,2,114,9807,1,0,2169975,0,4.200000,0,1.791759,6.398595,0,0,0,4.0,75.0,132.0
3,tr_caceaad574bb3bc4,48d6c714ed99e368,14.879358,10.040675,1.836444,10.268937,0.993527,200.0,10.0,0,1.0,1.0,0.15,1195.091064,0,1,1,1,3851,0,1,2.0,2,114,9807,1,0,2169983,1,4.921568,0,3.951244,5.707110,0,0,0,5.0,79.0,182.0
4,tr_caceaad574bb3bc4,44d055aa287eb7f7,2.654445,2.090788,1.947276,3.344653,0.177243,18.0,50.0,1,0.5,0.5,0.10,1.324593,1,1,1,1,3851,0,1,2.0,2,114,9807,1,0,2169973,0,4.866667,0,3.828641,6.175867,0,0,0,5.0,64.0,350.0
5,tr_caceaad574bb3bc4,653002c3a8449531,2.452737,2.304721,2.537263,2.678664,0.163775,187.0,50.0,1,0.5,0.5,0.00,6.681722,1,1,1,1,3851,0,1,2.0,2,114,9807,1,0,84438,0,4.902778,0,4.290460,0.000000,1,0,0,4.0,55.0,104.0
6,tr_caceaad574bb3bc4,9458f2aae4c01f30,2.579591,0.000000,2.320777,3.226665,0.172245,68.0,50.0,1,0.0,0.5,0.00,9.754232,1,1,1,1,3851,0,1,2.0,2,114,9807,1,0,2303705,0,5.000000,0,2.564949,6.216606,0,0,0,6.0,24.0,132.0
7,tr_caceaad574bb3bc4,ca6b6a63d43b06fb,2.488156,2.090788,1.615735,3.033416,0.166140,144.0,50.0,1,0.5,0.5,0.15,3.003438,1,1,1,1,3851,0,1,2.0,2,114,9807,1,0,2169983,1,4.958467,0,5.749393,6.216606,0,0,0,5.0,115.0,265.0
8,tr_caceaad574bb3bc4,2a1bda94e57a8b7d,2.668334,1.913198,1.924053,3.381814,0.178171,10.0,50.0,1,0.5,0.5,0.15,3.041692,1,1,1,1,3851,0,1,2.0,2,114,9807,1,0,2169976,1,4.840909,0,6.089045,0.000000,1,0,0,6.0,67.0,274.0
9,tr_caceaad574bb3bc4,cee71f8f28ad6ac7,2.567537,2.304721,2.070515,3.270529,0.171440,76.0,50.0,1,0.5,0.5,0.05,1.148797,1,1,1,1,3851,0,1,2.0,2,114,9807,1,0,1289835,0,5.000000,0,5.141664,9.615872,0,0,0,4.0,141.0,104.0


val: первые 10 строк


,qid,item_id,bm25_all,bm25_title,bm25_params,bm25_desc,bm25_rel,rank_pool,rank_global,in_pool,cov_title,cov_all,mc_share_top20,dist_km,same_loc,filter_text_ok,rating_ok,filter_ok,pool_size,target,n_pos_total,q_n_terms,q_len_words,search_category,loc_n_items,geo_known,has_filter,item_microcat_id,is_remote_text,item_rating,rating_isna,reviews_log,price_log,price_is_symbolic,item_is_phone_hidden,item_is_message_forbidden,title_len,params_len,desc_len
0,tr_da53db2835a7cdcf,33a7e4771a5d8853,8.021322,0.000000,0.000000,7.530546,0.370010,90.0,50.0,1,0.000000,0.666667,0.0,15.970729,1,0,1,0,14693,0,1,3.0,4,114,28507,1,1,2303428,0,4.680412,0,4.584968,0.000000,1,0,0,5.0,22.0,115.0
1,tr_da53db2835a7cdcf,048dabd0ee217c21,14.453650,0.000000,12.842342,9.124809,0.666723,56.0,50.0,1,0.000000,1.000000,1.0,14.276658,0,1,1,1,14693,0,1,3.0,4,114,28507,1,1,44728,0,5.000000,0,2.995732,6.396930,0,0,0,4.0,620.0,182.0
2,tr_da53db2835a7cdcf,7c34931b9fd2dbbf,16.558109,2.553852,6.071312,15.806046,0.763799,33.0,50.0,1,0.333333,1.000000,1.0,15.368428,1,1,1,1,14693,0,1,3.0,4,114,28507,1,1,44728,0,5.000000,0,3.295837,10.817797,0,0,0,7.0,173.0,289.0
3,tr_da53db2835a7cdcf,ef11d1ec8cbced91,6.385192,0.000000,4.294514,2.579087,0.294538,129.0,50.0,1,0.000000,0.666667,0.0,13.003131,1,0,1,0,14693,0,1,3.0,4,114,28507,1,1,2058735,0,4.940476,0,4.442651,5.525453,0,0,0,3.0,213.0,149.0
4,tr_da53db2835a7cdcf,5280e8d77a295be9,9.017302,0.000000,0.000000,8.798950,0.415953,78.0,50.0,1,0.000000,0.666667,0.0,11.090480,0,0,1,0,14693,0,1,3.0,4,114,28507,1,1,1288223,0,4.815217,0,4.532599,10.819798,0,0,0,4.0,101.0,246.0
5,tr_da53db2835a7cdcf,84fd6c4c3860d4bf,20.620707,14.454041,18.817001,20.282303,0.951200,200.0,33.0,0,1.000000,1.000000,1.0,624.337097,0,1,1,1,14693,0,1,3.0,4,114,28507,1,1,44728,0,4.872093,0,5.153292,6.908755,0,0,0,5.0,312.0,112.0
6,tr_da53db2835a7cdcf,127d4add6639840a,15.341412,0.000000,12.842342,12.144196,0.707674,45.0,50.0,1,0.000000,1.000000,1.0,23.994780,0,1,1,1,14693,0,1,3.0,4,114,28507,1,1,44728,0,5.000000,0,2.995732,6.216606,0,0,0,4.0,620.0,204.0
7,tr_da53db2835a7cdcf,3acec4bb65863027,7.314483,3.337795,2.964532,7.321335,0.337405,103.0,50.0,1,0.333333,0.666667,0.0,2.251493,1,0,1,0,14693,0,1,3.0,4,114,28507,1,1,44725,0,5.000000,0,2.079442,9.210441,0,0,0,4.0,164.0,214.0
8,tr_da53db2835a7cdcf,347a3be3dab07577,20.979952,14.030869,17.093277,21.402603,0.967771,200.0,15.0,0,0.666667,1.000000,1.0,713.367554,0,1,1,1,14693,0,1,3.0,4,114,28507,1,1,44728,0,4.964286,0,4.727388,6.908755,0,0,0,3.0,249.0,237.0
9,tr_da53db2835a7cdcf,e552ebf140a66c19,18.406183,0.000000,6.877143,20.211662,0.849047,19.0,50.0,1,0.000000,1.000000,1.0,15.482668,0,1,1,1,14693,0,1,3.0,4,114,28507,1,1,44728,0,4.882979,0,5.241747,8.243019,0,0,0,4.0,193.0,136.0


In [34]:
print('Статистика признаков: train vs val (среднее)')
cmp = pd.DataFrame({'train_mean': train_df[FEATURES].mean(numeric_only=True),
                    'val_mean': val_df[FEATURES].mean(numeric_only=True),
                    'train_nan_%': train_df[FEATURES].isna().mean() * 100,
                    'val_nan_%': val_df[FEATURES].isna().mean() * 100})
display(cmp.round(3))

Статистика признаков: train vs val (среднее)


,train_mean,val_mean,train_nan_%,val_nan_%
bm25_all,6.491,7.772,0.000,0.000
bm25_title,2.749,2.635,0.000,0.000
bm25_params,2.781,2.920,0.000,0.000
bm25_desc,5.863,6.900,0.000,0.000
bm25_rel,0.461,0.481,0.000,0.000
rank_pool,117.769,116.989,0.000,0.000
rank_global,44.725,44.696,0.000,0.000
in_pool,0.813,0.823,0.000,0.000
cov_title,0.259,0.192,0.000,0.000
cov_all,0.522,0.519,0.000,0.000


In [35]:
display(val_df.groupby('target')[FEATURES].mean(numeric_only=True).T.round(3))

target,0,1
bm25_all,7.740,15.775
bm25_title,2.615,7.726
bm25_params,2.910,5.498
bm25_desc,6.873,13.762
bm25_rel,0.480,0.863
rank_pool,117.372,20.326
rank_global,44.748,31.560
in_pool,0.823,0.962
cov_title,0.191,0.490
cov_all,0.517,0.883


In [36]:
ex_q = val_df.loc[val_df.target == 1, 'qid'].iloc[0]
qrow = queries.set_index('qid').loc[ex_q]
print(f'запрос: «{qrow.search_query}» | фильтр: «{qrow.search_infm_params_text}» | локация: {qrow.search_location_id}')
ex = (val_df[val_df.qid == ex_q].sort_values('bm25_all', ascending=False)
      .merge(items[['item_id', 'title_norm']], on='item_id'))
display(ex[['title_norm', 'target', 'bm25_all', 'bm25_title', 'cov_all', 'rank_pool', 'in_pool', 'dist_km',
            'same_loc', 'filter_ok', 'mc_share_top20', 'item_rating']].head(15))

запрос: «остекление лоджий по ключ» | фильтр: «Тип услуги Окна и балконы Вид услуги Ремонт и отделка» | локация: 653240


,title_norm,target,bm25_all,bm25_title,cov_all,rank_pool,in_pool,dist_km,same_loc,filter_ok,mc_share_top20,item_rating
0,Остекление балконов остекление лоджий под ключ,0,21.732519,16.812889,1.0,200.0,0,633.713989,0,1,1.0,5.000000
1,Остекление балконов и лоджий под ключ,0,21.678635,15.933002,1.0,0.0,1,23.994780,0,1,1.0,4.819445
2,Остекление балконов и лоджий под ключ,0,21.658718,15.933002,1.0,200.0,0,651.663025,0,1,1.0,4.819445
3,"Балкон под ключ, лоджия под ключ, остекление балкона и л...",0,21.620907,12.564341,1.0,200.0,0,1355.797241,0,1,1.0,5.000000
4,"Окна, Балконы, Остекление, Отделка балконов, Двери",0,21.381588,5.186008,1.0,200.0,0,1050.145020,0,1,1.0,4.915254
5,Отделка и остекление балконов под ключ СПБ и ло,0,21.262606,7.956783,1.0,1.0,1,11.090480,0,1,1.0,5.000000
6,Остекление балконов и лоджий под ключ,0,21.246700,15.933002,1.0,2.0,1,24.320650,0,1,1.0,5.000000
7,Остекление утепления ремонт балконов и лоджии,0,21.230391,11.426073,1.0,200.0,0,630.262939,0,1,1.0,4.909091
8,Теплое остекление балконов без сквозняков,0,21.227669,5.667394,1.0,200.0,0,644.856323,0,1,1.0,5.000000
9,Отделка балконов и лоджий под ключ,0,21.218248,9.685712,1.0,200.0,0,651.583252,0,1,1.0,5.000000


In [37]:
import os, re, gc, time, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

PREP_DIR = '/kaggle/working/prep' if os.path.isdir('/kaggle/working/prep') else './prep'
FEAT_DIR = PREP_DIR.replace('prep', 'features')
assert os.path.isfile(f'{PREP_DIR}/items.parquet'), f'нет {PREP_DIR}/items.parquet — сессия перезапускалась?'
assert os.path.isfile(f'{FEAT_DIR}/train.parquet'), f'нет {FEAT_DIR}/train.parquet'

SEED = 42
N_FOLDS = 4          # фолды для out-of-fold предсказаний на запросах CatBoost-train
TOP_N = 30           # сколько микрокатегорий на запрос сохраняем
MIN_CLASS_COUNT = 3  # микрокатегории с меньшим числом примеров не учим (шум)
ALPHA = 5e-6         # регуляризация SGD
MAX_ITER = 10
print(PREP_DIR, FEAT_DIR)

/kaggle/working/prep /kaggle/working/features


In [38]:
queries = pd.read_parquet(f'{PREP_DIR}/queries.parquet',
                          columns=['qid', 'split', 'query_key', 'query_norm', 'query_lem', 'search_infm_params_text'])
pairs = pd.read_parquet(f'{PREP_DIR}/pairs.parquet', columns=['qid', 'item_id', 'split'])
items = pd.read_parquet(f'{PREP_DIR}/items.parquet', columns=['item_id', 'item_microcat_id', 'title_norm'])
cb_train_qids = set(pd.read_parquet(f'{FEAT_DIR}/train.parquet', columns=['qid'])['qid'].unique())
val_qids = set(pd.read_parquet(f'{FEAT_DIR}/val.parquet', columns=['qid'])['qid'].unique())

mc_of_item = items.set_index('item_id')['item_microcat_id']
pairs['microcat'] = pairs['item_id'].map(mc_of_item)
print(f'запросов: {len(queries):,}, пар: {len(pairs):,}, объявлений: {len(items):,}')
print(f'CatBoost-train запросов: {len(cb_train_qids):,}, val: {len(val_qids):,}, bench: {(queries.split == "bench").sum():,}')

запросов: 356,914, пар: 467,054, объявлений: 515,895
CatBoost-train запросов: 15,000, val: 3,000, bench: 2,452


In [39]:
WS_RE = re.compile(r'\s+')
def filter_text(s):
    s = (s or '').lower().replace('ё', 'е')
    return WS_RE.sub(' ', re.sub(r'[^a-zа-я0-9]+', ' ', s)).strip()

queries['q_lem'] = queries['query_lem'].fillna('')
queries['q_chr'] = queries['query_norm'].fillna('').str.lower()
queries['q_flt'] = queries['search_infm_params_text'].fillna('').map(filter_text)

tr_texts = queries[queries.split == 'train']
t = time.time()
vec_word = TfidfVectorizer(token_pattern=r'\S+', ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(tr_texts.q_lem)
vec_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=3, sublinear_tf=True,
                           max_features=300_000).fit(tr_texts.q_chr)
vec_flt  = TfidfVectorizer(token_pattern=r'\S+', ngram_range=(1, 3), min_df=2, sublinear_tf=True).fit(tr_texts.q_flt)

def vectorize(df):
    return sp.hstack([vec_word.transform(df.q_lem), vec_char.transform(df.q_chr),
                      vec_flt.transform(df.q_flt)]).tocsr()

print(f'словари: слова {len(vec_word.vocabulary_):,}, символы {len(vec_char.vocabulary_):,}, '
      f'фильтр {len(vec_flt.vocabulary_):,} | {time.time() - t:.0f} с')

словари: слова 36,267, символы 61,513, фильтр 6,662 | 19 с


In [40]:
tr = (pairs[pairs.split == 'train'].dropna(subset=['microcat'])
      .merge(queries[['qid', 'query_key', 'q_lem', 'q_chr', 'q_flt']], on='qid'))
tr['microcat'] = tr['microcat'].astype('int64')
tr['is_cb'] = tr['qid'].isin(cb_train_qids)

class_counts = tr['microcat'].value_counts()
KEEP_CLASSES = class_counts[class_counts >= MIN_CLASS_COUNT].index
tr = tr[tr.microcat.isin(KEEP_CLASSES)]

def dedup(df):
    """Схлопывает одинаковые (текст, фильтр, микрокатегория), вес = число повторов."""
    return (df.groupby(['q_lem', 'q_chr', 'q_flt', 'microcat'], as_index=False)
              .agg(weight=('qid', 'size'), query_key=('query_key', 'first')))

print(f'пар train с микрокатегорией: {len(tr):,}, классов: {len(KEEP_CLASSES)} '
      f'(отброшено редких: {(class_counts < MIN_CLASS_COUNT).sum()})')
print(f'после схлопывания повторов: {len(dedup(tr)):,} строк')

пар train с микрокатегорией: 460,322, классов: 201 (отброшено редких: 11)
после схлопывания повторов: 129,197 строк


In [41]:
def fit_clf(df):
    d = dedup(df)
    clf = SGDClassifier(loss='log_loss', alpha=ALPHA, max_iter=MAX_ITER, tol=1e-3,
                        n_jobs=-1, random_state=SEED)
    t = time.time()
    clf.fit(vectorize(d), d['microcat'].values, sample_weight=d['weight'].values)
    print(f'  обучено на {len(d):,} строк за {time.time() - t:.0f} с')
    return clf

def predict_top(clf, qdf, top_n=TOP_N):
    """-> long-таблица qid, microcat, proba (топ-N микрокатегорий на запрос)."""
    P = clf.predict_proba(vectorize(qdf)).astype('float32')
    top_n = min(top_n, P.shape[1])
    top = np.argsort(-P, axis=1)[:, :top_n]
    return pd.DataFrame({
        'qid': np.repeat(qdf['qid'].values, top_n),
        'microcat': clf.classes_[top].ravel(),
        'proba': np.take_along_axis(P, top, axis=1).ravel(),
    })

rng = np.random.default_rng(SEED)
cb_q = queries[queries.qid.isin(cb_train_qids)]
cb_texts = cb_q['query_key'].unique()
fold_of_text = pd.Series(rng.integers(0, N_FOLDS, len(cb_texts)), index=cb_texts)

oof_parts = []
for k in range(N_FOLDS):
    fold_texts = set(fold_of_text[fold_of_text == k].index)
    print(f'фолд {k}: текстов {len(fold_texts):,}')
    clf_k = fit_clf(tr[~tr.query_key.isin(fold_texts)])
    oof_parts.append(predict_top(clf_k, cb_q[cb_q.query_key.isin(fold_texts)]))
    del clf_k; gc.collect()
oof = pd.concat(oof_parts, ignore_index=True)
print(f'OOF: {oof.qid.nunique():,} запросов CatBoost-train')

фолд 0: текстов 1,751
  обучено на 120,503 строк за 69 с
фолд 1: текстов 1,707
  обучено на 120,338 строк за 70 с
фолд 2: текстов 1,665
  обучено на 120,474 строк за 68 с
фолд 3: текстов 1,685
  обучено на 120,364 строк за 66 с
OOF: 15,000 запросов CatBoost-train


In [42]:
clf_full = fit_clf(tr)
vb_q = queries[queries.qid.isin(val_qids) | (queries.split == 'bench')]
full_pred = predict_top(clf_full, vb_q)
mc_proba = pd.concat([oof, full_pred], ignore_index=True)
print(f'итого: {mc_proba.qid.nunique():,} запросов × топ-{TOP_N} = {len(mc_proba):,} строк')

  обучено на 129,197 строк за 69 с
итого: 20,452 запросов × топ-30 = 613,560 строк


In [43]:
def topk_acc(pred, qids, split_pairs):
    truth = split_pairs[split_pairs.qid.isin(qids)][['qid', 'microcat']].dropna().drop_duplicates()
    ranked = pred[pred.qid.isin(qids)].copy()
    ranked['rank'] = ranked.groupby('qid')['proba'].rank(method='first', ascending=False)
    m = truth.merge(ranked, on=['qid', 'microcat'], how='left')
    best = m.groupby('qid')['rank'].min()             # лучшая позиция среди правильных микрокатегорий запроса
    return {f'top-{k}': float((best <= k).mean()) for k in [1, 3, 10, 30]}

acc = pd.DataFrame({
    'CatBoost-train (OOF)': topk_acc(mc_proba, cb_train_qids, pairs),
    'val': topk_acc(mc_proba, val_qids, pairs),
})
display(acc.style.format('{:.2%}'))

,CatBoost-train (OOF),val
top-1,80.27%,65.93%
top-3,88.84%,79.80%
top-10,92.03%,84.63%
top-30,93.18%,86.33%


In [44]:
mc_example = (items.dropna(subset=['item_microcat_id']).groupby('item_microcat_id')['title_norm']
              .agg(lambda s: s.value_counts().index[0]))
ex_q = queries[queries.qid.isin(val_qids)].sample(8, random_state=1)
for _, r in ex_q.iterrows():
    top3 = mc_proba[mc_proba.qid == r.qid].nlargest(3, 'proba')
    truth = set(pairs.loc[pairs.qid == r.qid, 'microcat'].dropna().astype('int64'))
    print(f'\n«{r.query_norm}» | фильтр: «{r.search_infm_params_text or ""}»')
    for _, t_ in top3.iterrows():
        mark = '✓' if t_.microcat in truth else ' '
        print(f'   {mark} {t_.proba:.2f}  {t_.microcat}: {str(mc_example.get(t_.microcat, "?"))[:60]}')


«аренда автобуса с водителем 35 мест» | фильтр: «Вид услуги Пассажирские перевозки»
   ✓ 0.99  2097894: Пассажирские перевозки
     0.00  2097893: Трансфер
     0.00  4141: Юрист

«доставка таобао» | фильтр: «»
   ✓ 0.86  2303375: Байер в Китае. Есть своя база поставщиков
     0.08  2097892: Грузоперевозки 5 10 20 т фура газель
     0.02  2303381: Аренда электровелосипедов для курьеров

«скупка паллет» | фильтр: «»
     0.26  2303381: Аренда электровелосипедов для курьеров
     0.18  4145: Бизнес план для соцконтракта
     0.08  2097897: Фулфилмент для маркетплейсов под ключ

«госномер изготовление» | фильтр: «Вид услуги Оборудование, производство»
   ✓ 0.96  2303428: Изготовление дубликатов гос номеров
     0.02  2303426: Токарные работы
     0.01  2303427: Лазерная резка металла

«настройка роутера keenetic» | фильтр: «Вид услуги Компьютерная помощь»
   ✓ 0.88  2097578: Подключить домашний интернет wifi и тв Ростелеком
     0.07  2097577: Установка любых программ и игр на MacOS и Wi

In [45]:
mc_proba['microcat'] = mc_proba['microcat'].astype('int64')
mc_proba.to_parquet(f'{FEAT_DIR}/microcat_proba.parquet', index=False)
print(f'{FEAT_DIR}/microcat_proba.parquet: {os.path.getsize(f"{FEAT_DIR}/microcat_proba.parquet") / 2**20:.1f} МБ')
display(mc_proba.head(10))

/kaggle/working/features/microcat_proba.parquet: 3.5 МБ


,qid,microcat,proba
0,tr_0064dc680a264a20,1289831,0.703910
1,tr_0064dc680a264a20,2303374,0.270672
2,tr_0064dc680a264a20,4145,0.002340
3,tr_0064dc680a264a20,1178215,0.001788
4,tr_0064dc680a264a20,2303381,0.001746
5,tr_0064dc680a264a20,86455,0.001075
6,tr_0064dc680a264a20,2303375,0.000922
7,tr_0064dc680a264a20,2303378,0.000869
8,tr_0064dc680a264a20,2303428,0.000866
9,tr_0064dc680a264a20,2303105,0.000804
